In [ ]:
!pip install numpy==1.26.4 --quiet
!pip install torch torchvision torchaudio --quiet
!pip install torch-geometric --quiet
!pip install networkx scikit-learn --quiet


In [ ]:
import torch


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
import glob
import logging
import os
import shutil
import numpy as np
import torch
from torch_geometric.datasets import Reddit
from torch_geometric.data import Data

# Match training/train_reddit.py: subsample, normalize, 4-class labels.
# Prefer local Kaggle .npz add-on; fallback to PyG Reddit (downloads) if npz not present.
REDDIT_ROOT = os.environ.get("REDDIT_DATA_ROOT", "/kaggle/working/reddit_data")
REDDIT_NPZ_SOURCE = os.environ.get(
    "REDDIT_NPZ_SOURCE", "/kaggle/input/datasets/akshatsharma1011/reddit-files"
)
NPZ_DIR = os.path.join(REDDIT_ROOT, "npz_kaggle")
REDDIT_MAX_NODES = int(os.environ.get("REDDIT_MAX_NODES", "250000"))
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)


def prepare_reddit_npz() -> int:
    """Copy reddit_*.npz from read-only Kaggle input into REDDIT_ROOT/npz_kaggle."""
    if not os.path.isdir(REDDIT_NPZ_SOURCE):
        log.info("Kaggle reddit-files folder not found: %s", REDDIT_NPZ_SOURCE)
        return 0
    os.makedirs(NPZ_DIR, exist_ok=True)
    n = 0
    for p in glob.glob(os.path.join(REDDIT_NPZ_SOURCE, "*.npz")):
        if os.path.isfile(p):
            shutil.copy2(p, os.path.join(NPZ_DIR, os.path.basename(p)))
            n += 1
    log.info("Copied %d .npz file(s) -> %s", n, NPZ_DIR)
    return n


def _pick(arrs: dict, *names, required=True):
    for n in names:
        if n in arrs and arrs[n] is not None:
            return arrs[n]
    if required:
        raise KeyError(f"None of {names} in npz; available: {list(arrs.keys())}")
    return None


def _to_dense_features(feat, num_nodes_hint=None):
    if feat is None:
        return None
    if isinstance(feat, np.ndarray) and feat.dtype == object and feat.size == 1:
        feat = feat.item()
    try:
        from scipy import sparse as sp
        if sp.issparse(feat):
            feat = feat.toarray()
    except Exception:
        pass
    a = np.asarray(feat, dtype=np.float32)
    if a.ndim == 1 and num_nodes_hint and num_nodes_hint > 0:
        if a.size == num_nodes_hint:
            a = a.reshape(-1, 1)
        elif a.size % num_nodes_hint == 0 and a.size != num_nodes_hint:
            a = a.reshape(num_nodes_hint, -1)
    return a


def _build_edge_index(merged: dict) -> np.ndarray:
    ei = _pick(merged, "edge_index", "edges", required=False)
    if ei is not None:
        ei = np.asarray(ei, dtype=np.int64)
        if ei.shape[0] != 2 and ei.size and ei.shape[1] == 2:
            ei = ei.T
        return ei
    if "row" in merged and "col" in merged:
        r = np.asarray(merged["row"], dtype=np.int64).ravel()
        c = np.asarray(merged["col"], dtype=np.int64).ravel()
        if r.size != c.size:
            raise ValueError(f"row {r.size} vs col {c.size}")
        return np.stack([r, c], axis=0)
    raise KeyError(f"No edge_index/edges or row+col; keys: {list(merged.keys())}")


def load_reddit_from_npz():
    gfile = os.path.join(NPZ_DIR, "reddit_graph.npz")
    dfile = os.path.join(NPZ_DIR, "reddit_data.npz")
    if not (os.path.isfile(gfile) and os.path.isfile(dfile)):
        return None
    merged = {}
    for path in (gfile, dfile):
        z = np.load(path, allow_pickle=True)
        for k in z.files:
            merged[k] = z[k]
    y = _pick(merged, "y", "Y", "labels", "label", "y_train", "y_all")
    y = np.asarray(y).reshape(-1).astype(np.int64)
    num_nodes = int(y.shape[0])
    x = _pick(
        merged, "x", "X", "features", "feat", "feature", "Feature", "node_features", required=False
    )
    if x is None:
        raise KeyError(f"No node features; keys: {list(merged.keys())}")
    x = _to_dense_features(x, num_nodes)
    if x.shape[0] != num_nodes:
        if x.ndim == 2 and x.shape[1] == num_nodes:
            x = x.T
        else:
            raise ValueError(f"feature rows {x.shape[0]} != num_nodes {num_nodes}")
    ei = _build_edge_index(merged)
    ei = np.asarray(ei, dtype=np.int64)
    if ei.shape[0] != 2:
        ei = ei.T if ei.shape[1] == 2 else ei
    return Data(
        x=torch.from_numpy(x),
        y=torch.from_numpy(y),
        edge_index=torch.from_numpy(ei).long().contiguous(),
        num_nodes=num_nodes,
    )


def normalize_features(x):
    norm = x.norm(p=1, dim=1, keepdim=True).clamp(min=1.0)
    return x / norm


def subsample_graph(data, max_nodes: int) -> Data:
    n = int(data.num_nodes)
    if n <= max_nodes:
        return data
    dev = data.x.device
    perm = torch.randperm(n, device=dev)[:max_nodes]
    inv = torch.full((n,), -1, dtype=torch.long, device=dev)
    inv[perm] = torch.arange(max_nodes, device=dev, dtype=torch.long)
    e = data.edge_index.to(dev)
    src, dst = e[0], e[1]
    ok = (inv[src] >= 0) & (inv[dst] >= 0)
    new_ei = torch.stack([inv[src[ok]], inv[dst[ok]]], dim=0).contiguous()
    return Data(
        x=data.x[perm],
        y=data.y[perm],
        edge_index=new_ei,
        num_nodes=max_nodes,
    )


prepare_reddit_npz()
data_from_npz = load_reddit_from_npz()
if data_from_npz is not None:
    log.info("Using Reddit from npz (reddit_data.npz + reddit_graph.npz).")
    data_raw = data_from_npz
else:
    log.info("Loading PyG Reddit (downloads into REDDIT_ROOT if needed)...")
    data_raw = Reddit(root=REDDIT_ROOT)[0]
if data_raw.num_nodes > REDDIT_MAX_NODES:
    log.info("Subsample %d -> %d nodes (REDDIT_MAX_NODES)", data_raw.num_nodes, REDDIT_MAX_NODES)
    data_raw = subsample_graph(data_raw, REDDIT_MAX_NODES)
data_raw.x = normalize_features(data_raw.x)
original_classes = int(data_raw.y.max().item()) + 1
data_raw.y = (data_raw.y * 4 // original_classes).clamp(0, 3)
X = data_raw.x.numpy()
Y = data_raw.y.numpy().astype(int)
edge_index = data_raw.edge_index.contiguous()
print("Feature matrix shape:", X.shape, "| edges (directed as stored):", edge_index.shape[1])


In [ ]:
from collections import Counter
counts = Counter(Y)
print(counts.most_common(10))


In [ ]:
# Node ids are 0..N-1; edge_index already uses local indices
print("Edge index shape:", edge_index.shape)


In [ ]:
from torch_geometric.data import Data

x = torch.tensor(X, dtype=torch.float)
y = torch.tensor(Y, dtype=torch.long)

data = Data(x=x, edge_index=edge_index, y=y)
data = data.to(device)
print(data)


In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
import numpy as np
import torch

counts = Counter(Y)
valid_classes = {cls for cls, cnt in counts.items() if cnt >= 2}
valid_indices = [i for i, label in enumerate(Y) if label in valid_classes]
if len(valid_indices) == 0:
    raise ValueError("No valid classes for stratified split; check labels.")

Y_filtered = np.array([Y[i] for i in valid_indices])
if len(set(Y_filtered)) < 2:
    train_idx, test_idx = train_test_split(valid_indices, test_size=0.2, random_state=42)
else:
    train_idx, test_idx = train_test_split(
        valid_indices, test_size=0.2, stratify=Y_filtered, random_state=42
    )

train_mask = torch.zeros(len(Y), dtype=torch.bool)
test_mask = torch.zeros(len(Y), dtype=torch.bool)
train_mask[train_idx] = True
test_mask[test_idx] = True
data.train_mask = train_mask
data.test_mask = test_mask
print("Train size:", train_mask.sum().item(), "| Test size:", test_mask.sum().item())


In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

model = GraphSAGE(data.num_features, 64, len(set(Y.tolist()))).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


In [ ]:
import torch.nn.functional as F

batch_size = 1024
train_indices = data.train_mask.nonzero(as_tuple=True)[0]

def train():
    model.train()
    total_loss = 0.0
    perm = train_indices[torch.randperm(len(train_indices))]
    steps = 0
    for i in range(0, len(perm), batch_size):
        batch_nodes = perm[i : i + batch_size]
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[batch_nodes], data.y[batch_nodes])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        steps += 1
    return total_loss / max(steps, 1)

for epoch in range(20):
    loss = train()
    print(f"Epoch {epoch}, Loss: {loss:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score

def f1_auc_test(y_true, logits):
    y_true = y_true.cpu().numpy()
    proba = torch.softmax(logits, dim=1).cpu().numpy()
    pred = logits.argmax(dim=1).cpu().numpy()
    f1 = f1_score(y_true, pred, average="macro", zero_division=0)
    c = proba.shape[1]
    if c < 2:
        return f1, float("nan")
    try:
        if c == 2:
            auc = roc_auc_score(y_true, proba[:, 1])
        else:
            auc = roc_auc_score(
                y_true, proba, multi_class="ovr", average="macro", labels=np.arange(c)
            )
    except ValueError:
        auc = float("nan")
    return f1, auc


@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    return acc, pred, out

acc, pred, gnn_output = test()
f1, auc = f1_auc_test(data.y[data.test_mask], gnn_output[data.test_mask])
print(f"Test Accuracy: {acc:.4f} | F1 (macro): {f1:.4f} | AUC (macro OVR): {auc:.4f}")


In [ ]:
print("Unique predictions:", pred.unique())
print("Unique labels:", data.y.unique())
